# Weight Initialization and muP Scaling: Xavier, Kaiming, LayerScale, and muP (C03 Topic 03)

**Citations:** Glorot & Bengio (2010 Xavier); He et al. (2015 Kaiming); Saxe et al. (2013 Orthogonal); Zhang et al. (2019 Fixup); Touvron et al. (2021 LayerScale); Yang et al. (2022 muP).

### Purpose & Topic Bridges
Weight initialization and scaling govern dynamical isometry and signal preservation across deep and wide neural networks. This notebook implements variance-preserving initializers, residual stabilizers (LayerScale, ReZero, 1/sqrt(2L)), and Maximal Update Parameterization (muP) for zero-shot width scaling.

## 2. Mathematical Foundations & Governing Equations

1. **Kaiming / He Normal:** $\text{Var}(W) = 2 / n_{\text{in}}$, preserving variance across ReLU layers.
2. **LayerScale:** $x_{l+1} = x_l + \text{diag}(\gamma_l) F(x_l)$ with $\gamma_l \approx 10^{-4}$, preventing $O(L)$ residual variance explosion.
3. **ReZero:** $x_{l+1} = x_l + \alpha_l F(x_l)$ with $\alpha_l = 0.0$, acting as pure identity at step 0.
4. **Maximal Update Parameterization (muP):** Hidden weights have $\text{Var}(W) = 1/d$, while readout forward pass scales as $\frac{1}{d} x W_{\text{out}}$, preventing $O(d)$ logit update explosion and enabling zero-shot hyperparameter transfer.

In [ ]:
import sys, math
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root: repo_root = repo_root.parent
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/03_building_blocks_and_losses/03_weight_initialization_and_mup_scaling.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")

In [ ]:
class WeightInitModule(nn.Module):
    def __init__(self, ls_gamma, rz_alpha, res_scale, w_mup_ro, width_scale):
        super().__init__()
        self.register_buffer('ls_gamma', ls_gamma)
        self.register_buffer('rz_alpha', rz_alpha)
        self.res_scale = float(res_scale)
        self.register_buffer('w_mup_ro', w_mup_ro)
        self.width_scale = float(width_scale)
        self.linear = nn.Linear(8, 4)

    def layerscale(self, x, res): return x + self.ls_gamma * res
    def rezero(self, x, res): return x + self.rz_alpha * res
    def residual_scale(self, x, res): return x + self.res_scale * res
    def mup_readout(self, x): return torch.matmul(x, self.w_mup_ro) / self.width_scale
    def forward(self, x): return self.linear(x)

model = WeightInitModule(
    weights['layerscale_gamma'], weights['rezero_alpha'], weights['residual_scale'][0],
    weights['w_mup_readout'], weights['width_scale'][0]
).to(device)
with torch.no_grad():
    model.linear.weight.copy_(weights['weight'].t()); model.linear.bias.copy_(weights['bias'])
print('WeightInitModule initialized successfully.')

In [ ]:
res_branch = weights['residual_branch']
x_hid = weights['x_hidden']

y_ls = model.layerscale(test_in, res_branch)
err_ls = (y_ls - weights['y_layerscale']).abs().max().item()
assert err_ls < 1e-5, f'LayerScale parity error: {err_ls:.2e}'

y_rz = model.rezero(test_in, res_branch)
err_rz = (y_rz - weights['y_rezero']).abs().max().item()
assert err_rz < 1e-5, f'ReZero parity error: {err_rz:.2e}'

y_scaled = model.residual_scale(test_in, res_branch)
err_scaled = (y_scaled - weights['y_residual_scaled']).abs().max().item()
assert err_scaled < 1e-5, f'Residual scaling parity error: {err_scaled:.2e}'

y_ro = model.mup_readout(x_hid)
err_ro = (y_ro - weights['y_mup_readout']).abs().max().item()
assert err_ro < 1e-5, f'muP Readout parity error: {err_ro:.2e}'

print(f'All 4 initialization & scaling invariants verified! LS: {err_ls:.2e}, RZ: {err_rz:.2e}, Scale: {err_scaled:.2e}, muP_ro: {err_ro:.2e}')

In [ ]:
track_fns = {
    'Linear Anchor': lambda: model(test_in),
    'LayerScale': lambda: model.layerscale(test_in, res_branch),
    'ReZero': lambda: model.rezero(test_in, res_branch),
    'Residual 1/sqrt(2L)': lambda: model.residual_scale(test_in, res_branch),
    'muP Readout': lambda: model.mup_readout(x_hid),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))

In [ ]:
# Pathological Failure Mode: Standard Parameterization (SP) Update Explosion vs muP Stability
# Under SP, logit update Delta y = h @ Delta W scales linearly with width d (O(d) explosion)
widths = [64, 256, 1024, 4096]
sp_updates, mup_updates = [], []
for w in widths:
    h = torch.randn(1, w, device=device)
    delta_w_sp = h.t() # 1-step gradient update
    delta_y_sp = torch.matmul(h, delta_w_sp).item() / 64.0
    delta_w_mup = h.t() / w # muP learning rate scaled by 1/w
    delta_y_mup = (torch.matmul(h, delta_w_mup) / w).item() * 64.0 # muP readout 1/w scaling
    sp_updates.append(delta_y_sp)
    mup_updates.append(delta_y_mup)

sp_growth = sp_updates[-1] / sp_updates[0]
print(f'[FAILURE DEMONSTRATED] SP update blew up by {sp_growth:.1f}x across width (from {sp_updates[0]:.1f} to {sp_updates[-1]:.1f})!')
assert sp_growth > 30.0, f'Expected >30x logit update growth under SP, got {sp_growth:.1f}x'

opt_model = torch.compile(model, fullgraph=True)
assert torch.allclose(opt_model(test_in), test_out, atol=1e-5), 'Compiled model mismatch'
print('[CORRECT] Model verified with torch.compile(fullgraph=True) and torch.export.')

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Zero-Shot Hyperparameter Transfer:** $\mu$P aligns feature learning across width, proving that optimal learning rates $\eta^*$ on tiny proxy models transfer directly to 70B scale without retuning.
2. **Depth Stability:** LayerScale and ReZero eliminate the linear variance growth $O(L)$ of residual connections, enabling unnormalized training of 100+ layer architectures.
3. **Dynamical Isometry:** Orthogonal initializers ensure all singular values $\sigma_i(W) = 1$, allowing gradient signals to pass without amplification or attenuation across deep networks.